In [1]:
import os
import numpy as np
import pandas as pd
import math
from scipy import stats

from binlib import output_bin, calculate_r_with_groundtruth

ref_genome_folder = 'ref_genome/hg19'
exclude_chr = ['chrM', 'chrY','chrMT']
resolution = 50000


# load ref genome length info
chr_length = dict()
with open(ref_genome_folder + '/hg19.len', 'r') as f:
    lines = f.read().splitlines()
    for line in lines:
        chr_name , len_ = line.split('\t')[:2]
        if chr_name in exclude_chr:
            continue
        chr_length[chr_name] = int(len_)
print('chr length : \n',chr_length)

# calculate bin number in each chr
chr_bin_number = dict()
for key in chr_length.keys():
    chr_bin_number[key] = math.floor(chr_length[key] / resolution) + 1
print('chr bin number : \n', chr_bin_number)

Pearson r :  0.503
Spearman r :  0.683
chr length : 
 {'chr1': 249250621, 'chr2': 243199373, 'chr3': 198022430, 'chr4': 191154276, 'chr5': 180915260, 'chr6': 171115067, 'chr7': 159138663, 'chr8': 146364022, 'chr9': 141213431, 'chr10': 135534747, 'chr11': 135006516, 'chr12': 133851895, 'chr13': 115169878, 'chr14': 107349540, 'chr15': 102531392, 'chr16': 90354753, 'chr17': 81195210, 'chr18': 78077248, 'chr19': 59128983, 'chr20': 63025520, 'chr21': 48129895, 'chr22': 51304566, 'chrX': 155270560}
chr bin number : 
 {'chr1': 4986, 'chr2': 4864, 'chr3': 3961, 'chr4': 3824, 'chr5': 3619, 'chr6': 3423, 'chr7': 3183, 'chr8': 2928, 'chr9': 2825, 'chr10': 2711, 'chr11': 2701, 'chr12': 2678, 'chr13': 2304, 'chr14': 2147, 'chr15': 2051, 'chr16': 1808, 'chr17': 1624, 'chr18': 1562, 'chr19': 1183, 'chr20': 1261, 'chr21': 963, 'chr22': 1027, 'chrX': 3106}


### load IS

In [13]:
IS_path = 'insulation_score/GSE63525_K562_50k_IS_150kb.bed'

IS_dict = dict()
for key in chr_bin_number.keys():
    IS_dict[key] = np.zeros(chr_bin_number[key])

with open(IS_path, 'r') as f:
    for line in f.readlines():
        line_split = line.strip().split()
        chr_ = 'chr'+line_split[0]
        if chr_ in exclude_chr :
            continue
        bin_ = math.floor(int(line_split[1]) / resolution)
        if (line_split[4] == 'inf') or (line_split[4] == '-inf') or (line_split[4] == 'nan'):
            IS_dict[chr_][bin_] = 0
        elif (float(line_split[4]) > 1):  #remove outlier
            IS_dict[chr_][bin_] = 1
        elif (float(line_split[4]) < -1): #remove outlier
            IS_dict[chr_][bin_] = -1
        else:
            IS_dict[chr_][bin_] = float(line_split[4])

### load HiNT bin file

In [9]:
bin_dir = 'K562_HIC_CNV/HiNTcnv_OUTPUT_K562/segmentation/b50000'

bin_dict = dict()
for key in chr_bin_number.keys():
    bin_dict[key] = np.zeros(chr_bin_number[key])

keys = list(chr_bin_number.keys())
key_num = 0
for file in os.listdir(bin_dir):
    with open(f'{bin_dir}/{file}', 'r') as f:
        for line in f.readlines()[1:]:
            start_, end_, obs, exp ,var = line.strip().split()
            residual = float(obs) - float(exp)
            bin_ = math.floor(int(start_)/resolution)
            bin_dict[keys[key_num]][bin_] = residual
    key_num = key_num +1


### direct add both

In [15]:
bin_IS_dict = dict()
for key in chr_bin_number.keys():
    bin_IS_dict[key] = np.zeros(chr_bin_number[key])

for key in chr_bin_number.keys():
    bin_IS_dict[key] = bin_dict[key] + IS_dict[key]

In [33]:
output_bin(bin_IS_dict,'bin_combined_with_IS')

### calculate r

In [2]:
calculate_r_with_groundtruth('bin_combined_with_IS/seg_bin_conbined_with_IS.txt')

Pearson r :  0.503
Spearman r :  0.683
